<a href="https://colab.research.google.com/github/pierrot73/GenAIBootCamp/blob/Bootcamp/Week_7_Day1_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/train.csv")

In [ ]:

#Étape 0 : Préparation de l’environnement
# Installe les bibliothèques nécessaires
!pip install -q transformers datasets evaluate scikit-learn accelerate tensorflow
#Étape 1 : Chargement et Préparation des Données
import pandas as pd

# Charger les données
train_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test.csv')

# Afficher les premières lignes
print(train_df.head())
#Étape 2 : Exploration des Données (EDA)
# Distribution des labels
print(train_df['label'].value_counts())

import random

idx = random.randint(0, len(train_df) - 1)
print("Prémisse :", train_df.loc[idx, 'premise'])
print("Hypothèse :", train_df.loc[idx, 'hypothesis'])
print("Label :", train_df.loc[idx, 'label'])
# Étape 3 : Tokenization avec BERT
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def bert_encode(premises, hypotheses, tokenizer, max_length=128):
    return tokenizer(
        premises.tolist(),
        hypotheses.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

train_encodings = bert_encode(train_df['premise'], train_df['hypothesis'], tokenizer)
test_encodings = bert_encode(test_df['premise'], test_df['hypothesis'], tokenizer)

# Convertir les labels
import torch
train_labels = torch.tensor(train_df['label'].values)
#Étape 4 : Définition du Modèle de Classification
import torch.nn as nn
from transformers import BertModel

class BERTClassifier(nn.Module):
    def __init__(self):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)
#Étape 5 : Préparation à l’Entraînement
from torch.utils.data import TensorDataset, DataLoader

# Dataset
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    train_encodings['token_type_ids'],
    train_labels
)

# DataLoader
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16)

# Modèle
model = BERTClassifier()

# Optimiseur et perte
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# GPU si dispo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
#Étape 6 : Boucle d’Entraînement
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        input_ids, attention_mask, token_type_ids, labels = [b.to(device) for b in batch]

        outputs = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} | Loss: {total_loss / len(train_dataloader):.4f}")
#Étape 7 : Évaluation sur les Données de Test (Inférence)
from torch.utils.data import TensorDataset, DataLoader

# Pas de labels ici
test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    test_encodings['token_type_ids']
)

test_dataloader = DataLoader(test_dataset, batch_size=16)

model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids, attention_mask, token_type_ids = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask, token_type_ids)
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

# Création du fichier de soumission
submission = test_df.copy()
submission['label'] = all_preds
submission.to_csv('submission.csv', index=False)




           id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  
label
0    4176
2    4064
1    3880
Name: count, dtype: int64
Prémisse : okay okay that's it that GTE had purch

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]